In [1]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Modul 4a: Klasifikasi

Cerita lengkapnya ada di `README.md`. Hitung dulu, baru jalankan kode.

Paket adalah bundel kode siap pakai. Pustaka adalah kumpulan fungsi di dalam paket. pandas dan scikit-learn adalah pustaka yang dipakai notebook ini.

Dua contoh peluang, sebelum ada model:

1. Peluang virginica `0.7`, ambang `0.5`. Golongan mana?
2. Peluang virginica `0.4`, ambang `0.5`. Golongan mana?

Ambang adalah batas peluang yang kita pilih. Jawab dengan tangan, lalu jalankan sel dari atas.


In [2]:
# Satu ide: ambang 0.5 menentukan golongan pada dua contoh tangan.
ambang = 0.5
for peluang in (0.7, 0.4):
    if peluang > ambang:
        golongan = "virginica"
    else:
        golongan = "versicolor"
    print("peluang", peluang, "golongan", golongan)


peluang 0.7 golongan virginica
peluang 0.4 golongan versicolor


## Kasus nyata

Tabel bunga iris sudah ikut terpasang di scikit-learn. Sel di bawah tidak mengunduh apa pun. Ini bukan data asli perusahaan.

`sepal` artinya kelopak pelindung, `petal` artinya mahkota. Keduanya diukur dalam sentimeter. Spesies `setosa` mudah terpisah dari yang lain, jadi modul ini menyisihkannya dan memakai dua spesies yang lebih mirip: versicolor dan virginica.


In [3]:
# Satu ide: panggil pandas, pustaka untuk tabel.
import pandas as pd


In [4]:
# Satu ide: muat iris dari berkas yang sudah terpasang.
from sklearn.datasets import load_iris

iris = load_iris()


In [5]:
# Satu ide: simpan angka iris sebagai tabel, plus nama spesies.
# feature_names adalah nama kolom bawaan dataset, bukan istilah baru.
tabel = pd.DataFrame(iris.data, columns=iris.feature_names)
tabel["spesies"] = iris.target_names[iris.target]


In [6]:
# Satu ide: sisakan dua spesies yang lebih mirip, lalu hitung barisnya.
dua = tabel.loc[tabel["spesies"] != "setosa"].copy()
print("baris", int(dua.shape[0]))
print("kolom", int(dua.shape[1]))
print("versicolor", int((dua["spesies"] == "versicolor").sum()))
print("virginica", int((dua["spesies"] == "virginica").sum()))


baris 100
kolom 5
versicolor 50
virginica 50


In [7]:
# Satu ide: pisahkan kolom ukuran dan kolom golongan.
kolom_ukuran = list(iris.feature_names)
ukuran = dua[kolom_ukuran]
golongan = dua["spesies"]


In [8]:
# Satu ide: sisihkan data uji. Data latih untuk belajar; data uji tidak ikut belajar.
# test_size=0.25 berarti seperempat baris menjadi data uji.
# stratify menjaga proporsi tiap golongan di kedua bagian mirip dengan tabel penuh.
# random_state=SEED memakai angka tetap dari sel pertama.
from sklearn.model_selection import train_test_split

ukuran_latih, ukuran_uji, golongan_latih, golongan_uji = train_test_split(
    ukuran,
    golongan,
    test_size=0.25,
    random_state=SEED,
    stratify=golongan,
)
print("baris_latih", int(len(golongan_latih)))
print("baris_uji", int(len(golongan_uji)))


baris_latih 75
baris_uji 25


In [9]:
# Satu ide: baseline selalu menebak golongan yang paling banyak di data latih.
# DummyClassifier adalah nama baseline ini di scikit-learn.
# fit artinya model belajar dari data latih.
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent", random_state=SEED)
baseline.fit(ukuran_latih, golongan_latih)


DummyClassifier(random_state=42, strategy='most_frequent')

In [10]:
# Satu ide: hitung tebakan baseline yang salah.
# predict artinya model menebak golongan baris yang diberikan.
tebakan = baseline.predict(ukuran_uji)
salah_uji_baseline = int((tebakan != golongan_uji.to_numpy()).sum())
salah_latih_baseline = int(
    (baseline.predict(ukuran_latih) != golongan_latih.to_numpy()).sum()
)
print("baseline_selalu", tebakan[0])
print("salah_latih_baseline", salah_latih_baseline)
print("salah_uji_baseline", salah_uji_baseline)


baseline_selalu virginica
salah_latih_baseline 37
salah_uji_baseline 13


In [11]:
# Satu ide: pastikan baseline dan jumlah salah pada data uji.
assert baseline.strategy == "most_frequent", (
    "Baseline harus menebak golongan yang paling banyak."
)
assert salah_uji_baseline == 13, (
    "Salah uji baseline harus 13. Angka ini berubah bila strategi baseline diganti."
)


In [12]:
# Satu ide: latih regresi logistik pada data latih saja.
# random_state=SEED tetap ditulis. Pada solver bawaan, angka itu tidak mengubah hasil.
from sklearn.linear_model import LogisticRegression

logistik = LogisticRegression(random_state=SEED)
logistik.fit(ukuran_latih, golongan_latih)


LogisticRegression(random_state=42)

In [13]:
# Satu ide: lihat peluang baris pertama data uji, lalu tebakannya.
# iloc mengambil baris menurut urutan di tabel ini. Angka 0 berarti baris pertama.
# classes_ adalah daftar nama golongan menurut scikit-learn.
# predict_proba memberi peluang tiap golongan, antara 0 dan 1.
peluang_baris = logistik.predict_proba(ukuran_uji.iloc[[0]])[0]
for nama, nilai in zip(logistik.classes_, peluang_baris):
    print("peluang", nama, float(nilai))
peluang_virginica = float(
    peluang_baris[list(logistik.classes_).index("virginica")]
)
print("peluang_virginica", peluang_virginica)
print("di_atas_ambang", peluang_virginica > 0.5)
print("tebakan", logistik.predict(ukuran_uji.iloc[[0]])[0])


peluang versicolor 0.013309226417507314
peluang virginica 0.9866907735824927
peluang_virginica 0.9866907735824927
di_atas_ambang True
tebakan virginica


In [14]:
# Satu ide: bandingkan banyak tebakan salah regresi logistik dengan baseline.
tebakan_logistik = logistik.predict(ukuran_uji)
salah_uji_logistik = int((tebakan_logistik != golongan_uji.to_numpy()).sum())
salah_latih_logistik = int(
    (logistik.predict(ukuran_latih) != golongan_latih.to_numpy()).sum()
)
print("salah_latih_logistik", salah_latih_logistik)
print("salah_uji_logistik", salah_uji_logistik)
print("logistik_mengalahkan_baseline", salah_uji_logistik < salah_uji_baseline)


salah_latih_logistik 4
salah_uji_logistik 2
logistik_mengalahkan_baseline True


In [15]:
# Satu ide: latih pohon keputusan pendek, paling banyak dua pertanyaan beruntun.
# max_depth adalah batas banyak pertanyaan itu.
from sklearn.tree import DecisionTreeClassifier

pohon = DecisionTreeClassifier(max_depth=2, random_state=SEED)
pohon.fit(ukuran_latih, golongan_latih)


DecisionTreeClassifier(max_depth=2, random_state=42)

In [16]:
# Satu ide: cetak bagan pohon. feature_names di sini hanya nama kolom agar teksnya terbaca.
from sklearn.tree import export_text

print(export_text(pohon, feature_names=list(ukuran_latih.columns)))


|--- petal width (cm) <= 1.75
|   |--- petal length (cm) <= 5.05
|   |   |--- class: versicolor
|   |--- petal length (cm) >  5.05
|   |   |--- class: virginica
|--- petal width (cm) >  1.75
|   |--- class: virginica



In [17]:
# Satu ide: hitung tebakan salah pohon pendek di data latih dan data uji.
salah_uji_pohon = int(
    (pohon.predict(ukuran_uji) != golongan_uji.to_numpy()).sum()
)
salah_latih_pohon = int(
    (pohon.predict(ukuran_latih) != golongan_latih.to_numpy()).sum()
)
print("salah_latih_pohon", salah_latih_pohon)
print("salah_uji_pohon", salah_uji_pohon)
print("pohon_mengalahkan_baseline", salah_uji_pohon < salah_uji_baseline)
assert pohon.max_depth == 2, "Pohon pendek harus berhenti di dua pertanyaan."
assert salah_latih_pohon == 1, "Salah latih pohon pendek harus 1."
assert salah_uji_pohon == 3, (
    "Salah uji pohon pendek harus 3. Angka ini berubah bila batas kedalaman diganti."
)


salah_latih_pohon 1
salah_uji_pohon 3
pohon_mengalahkan_baseline True


## Pohon yang terlalu dalam

Overfitting, istilah dari modul regresi, tidak diubah di sini: model terlalu menyesuaikan diri dengan data latih, termasuk kebetulan-kebetulannya, sehingga bagus di data latih tetapi buruk di data baru. Sel berikut melepas batas pertanyaan.


In [18]:
# Satu ide: latih pohon tanpa batas kedalaman pada data latih yang sama.
pohon_dalam = DecisionTreeClassifier(random_state=SEED)
pohon_dalam.fit(ukuran_latih, golongan_latih)


DecisionTreeClassifier(random_state=42)

In [19]:
# Satu ide: hitung tebakan salah pohon dalam pada data latih dan data uji.
salah_uji_pohon_dalam = int(
    (pohon_dalam.predict(ukuran_uji) != golongan_uji.to_numpy()).sum()
)
salah_latih_pohon_dalam = int(
    (pohon_dalam.predict(ukuran_latih) != golongan_latih.to_numpy()).sum()
)
print("salah_latih_pohon_dalam", salah_latih_pohon_dalam)
print("salah_uji_pohon_dalam", salah_uji_pohon_dalam)
print("latih_nol", salah_latih_pohon_dalam == 0)
print(
    "uji_lebih_buruk_dari_pohon_pendek",
    salah_uji_pohon_dalam > salah_uji_pohon,
)


salah_latih_pohon_dalam 0
salah_uji_pohon_dalam 4
latih_nol True
uji_lebih_buruk_dari_pohon_pendek True


In [20]:
# Satu ide: angka salah uji pohon dalam harus lebih besar daripada regresi logistik.
assert salah_uji_logistik < salah_uji_pohon_dalam, (
    "Salah uji regresi logistik harus lebih kecil daripada pohon dalam."
)
assert salah_uji_logistik == 2, (
    "Salah uji regresi logistik harus 2, sama dengan angka di README."
)
assert salah_uji_pohon_dalam == 4, (
    "Salah uji pohon dalam harus 4, sama dengan angka di README."
)
